# 11 — Error Semantic Clustering

**Phase 5 E-2** — Cluster learner error types into semantic groups using TF-IDF + KMeans.

**Inputs:**
- `data/raw/attempt_*.csv` — attempt records with `error_type` column
- `data/features/sql_complexity_v1.parquet` — NB10 complexity features (prereq)

**Output:**
- `data/features/error_clusters_v1.parquet` — cluster label per attempt
- `data/features/cluster_manifest_v1.json` — schema/stats manifest

Research constraint: `label_validity = pilot_only` — results are pipeline-validation only.

In [ ]:
# cfg-01
from pathlib import Path
from datetime import datetime, timezone
import json, hashlib, warnings
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

warnings.filterwarnings('ignore')

SCHEMA_VERSION = 'cluster_v1'
N_CLUSTERS     = 5
RANDOM_STATE   = 42
MIN_ERROR_ROWS = 3          # skip clustering if fewer error rows than this

RAW_DIR      = Path('data/raw')
FEAT_DIR     = Path('data/features')
FEAT_DIR.mkdir(parents=True, exist_ok=True)

# Pin overrides — injected by run_e2e_notebooks.py
ATTEMPT_CSV : str | None = None

def newest_matching(pattern: str) -> Path | None:
    files = sorted(RAW_DIR.glob(pattern), key=lambda p: p.stat().st_mtime, reverse=True)
    return files[0] if files else None

def sha256_file(p: Path) -> str:
    h = hashlib.sha256()
    h.update(p.read_bytes())
    return h.hexdigest()[:16]

attempt_path = Path(ATTEMPT_CSV) if ATTEMPT_CSV else newest_matching('attempt_*.csv')
if attempt_path is None:
    raise FileNotFoundError('No attempt CSV found in data/raw/')

COMPLEXITY_PATH = FEAT_DIR / 'sql_complexity_v1.parquet'

print(f'Attempt CSV  : {attempt_path}')
print(f'Complexity   : {COMPLEXITY_PATH} (exists={COMPLEXITY_PATH.exists()})')
print(f'N_CLUSTERS   : {N_CLUSTERS}')
print(f'SCHEMA_VERSION: {SCHEMA_VERSION}')

## 2. Load attempt data

In [ ]:
# load-01
df = pd.read_csv(attempt_path, encoding='utf-8-sig')
print(f'Attempt rows: {len(df):,}  columns: {list(df.columns)}')

# Detect error column
ERROR_COL = None
for cand in ['error_type', 'error_message', 'error_code', 'error']:
    if cand in df.columns:
        ERROR_COL = cand
        break

if ERROR_COL is None:
    print('[WARN] No error column found — synthesising error_type from is_correct')
    df['error_type'] = df['is_correct'].apply(
        lambda x: 'no_error' if x else 'generic_error'
    )
    ERROR_COL = 'error_type'

print(f'Error column : {ERROR_COL}')
print(f'Non-null errors: {df[ERROR_COL].notna().sum():,}')
print(df[ERROR_COL].value_counts().head(10))

## 3. TF-IDF vectorisation + KMeans clustering

In [ ]:
# cluster-01
df_err = df[df[ERROR_COL].notna()].copy()
df_err['error_text'] = df_err[ERROR_COL].astype(str).str.lower().str.strip()

silhouette = None
cluster_centres = None
top_terms_per_cluster: dict[int, list[str]] = {}

if len(df_err) >= MIN_ERROR_ROWS:
    vec = TfidfVectorizer(
        analyzer='word', token_pattern=r'[a-z0-9_]+',
        min_df=1, max_features=200
    )
    X_tfidf = vec.fit_transform(df_err['error_text'])

    k = min(N_CLUSTERS, len(df_err) - 1)   # can't have more clusters than samples
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    df_err['cluster_id'] = km.fit_predict(X_tfidf)

    if len(df_err) > k:
        silhouette = float(silhouette_score(X_tfidf, df_err['cluster_id']))

    # Top terms per cluster
    feature_names = vec.get_feature_names_out()
    for cid in range(k):
        centre = km.cluster_centers_[cid]
        top_idx = centre.argsort()[-5:][::-1]
        top_terms_per_cluster[int(cid)] = [feature_names[i] for i in top_idx]

    print(f'KMeans k={k}  silhouette={silhouette:.4f}' if silhouette else f'KMeans k={k}')
    print('Cluster sizes:', df_err['cluster_id'].value_counts().sort_index().to_dict())
    for cid, terms in top_terms_per_cluster.items():
        print(f'  Cluster {cid}: {terms}')
else:
    print(f'[WARN] Too few error rows ({len(df_err)}) — assigning all to cluster 0')
    df_err['cluster_id'] = 0
    k = 1

## 4. Merge cluster labels back + summarise per learner × task

In [ ]:
# merge-01
# Per-attempt output
cols_keep = ['academy_member_id', 'task_id', 'attempt_no', ERROR_COL, 'cluster_id']
cols_keep = [c for c in cols_keep if c in df_err.columns]
df_attempt_clusters = df_err[cols_keep].copy()
df_attempt_clusters.rename(columns={ERROR_COL: 'error_type'}, inplace=True)

# Per (learner × task) summary
if 'academy_member_id' in df_err.columns and 'task_id' in df_err.columns:
    df_summary = (
        df_err
        .groupby(['academy_member_id', 'task_id'])
        .agg(
            error_count          = (ERROR_COL, 'count'),
            dominant_cluster     = ('cluster_id', lambda x: x.mode().iloc[0] if len(x) > 0 else -1),
            cluster_diversity    = ('cluster_id', 'nunique'),
        )
        .reset_index()
    )
    # Attach cluster label name
    df_summary['dominant_cluster_terms'] = df_summary['dominant_cluster'].apply(
        lambda c: '|'.join(top_terms_per_cluster.get(int(c), ['unknown']))
    )
else:
    df_summary = df_attempt_clusters.copy()

print(f'Per-attempt cluster rows : {len(df_attempt_clusters):,}')
print(f'Per-learner×task summary : {len(df_summary):,}')
print(df_summary.head(5).to_string())

In [ ]:
# viz-01
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

REPORTS_DIR = Path('reports/phase4')
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

if len(df_err) >= MIN_ERROR_ROWS:
    fig, ax = plt.subplots(figsize=(8, 4))
    counts = df_err['cluster_id'].value_counts().sort_index()
    bars = ax.bar([f'C{i}' for i in counts.index], counts.values,
                  color='#F37021', edgecolor='white')
    for bar, v in zip(bars, counts.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                str(v), ha='center', va='bottom', fontsize=9)
    ax.set_title('Error Cluster Distribution (NB11)', fontsize=12, fontweight='bold')
    ax.set_xlabel('Cluster ID')
    ax.set_ylabel('Error Count')
    fig.tight_layout()
    out_png = REPORTS_DIR / 'nb11_error_clusters.png'
    fig.savefig(out_png, dpi=150)
    plt.close(fig)
    print(f'Plot saved: {out_png}')
else:
    print('[skip] Not enough error rows for cluster plot')

## 5. Save artifacts

In [ ]:
# artifacts-01
CLUSTER_PATH   = FEAT_DIR / 'error_clusters_v1.parquet'
MANIFEST_PATH  = FEAT_DIR / 'cluster_manifest_v1.json'

df_summary.to_parquet(CLUSTER_PATH, index=False)

manifest = {
    'schema_version': SCHEMA_VERSION,
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'input_files': {
        'attempt_csv': str(attempt_path),
        'attempt_sha': sha256_file(attempt_path),
    },
    'parameters': {
        'n_clusters': N_CLUSTERS,
        'random_state': RANDOM_STATE,
        'error_column': ERROR_COL,
        'tfidf_max_features': 200,
    },
    'dataset_stats': {
        'total_attempt_rows': int(len(df)),
        'error_rows': int(len(df_err)),
        'n_clusters_actual': int(k),
        'silhouette_score': silhouette,
        'top_terms_per_cluster': top_terms_per_cluster,
        'summary_rows': int(len(df_summary)),
    },
    'data_warning': 'PILOT ONLY — proxy_behavioral labels, label_validity=pilot_only.',
    'artifacts': {
        'error_clusters': str(CLUSTER_PATH),
    },
}
MANIFEST_PATH.write_text(json.dumps(manifest, indent=2, default=str))

print(f'error_clusters_v1.parquet : {CLUSTER_PATH.stat().st_size:,} bytes')
print(f'cluster_manifest_v1.json  : {MANIFEST_PATH.stat().st_size:,} bytes')

## 6. Validation

In [ ]:
# validate-01
checks = []

def chk(name: str, passed: bool, detail: str = '') -> None:
    icon = '[OK]' if passed else '[FAIL]'
    checks.append({'name': name, 'passed': passed, 'detail': detail})
    print(f'{icon}  {name}' + (f' — {detail}' if detail else ''))

chk('parquet exists',        CLUSTER_PATH.exists(),              str(CLUSTER_PATH))
chk('parquet non-empty',     len(df_summary) > 0,               f'{len(df_summary)} rows')
chk('manifest exists',       MANIFEST_PATH.exists(),            str(MANIFEST_PATH))
chk('n_clusters valid',      k >= 1,                            f'k={k}')
chk('cluster_id column',     'dominant_cluster' in df_summary.columns, str(df_summary.columns.tolist()))
chk('cluster_diversity >=0', (df_summary['cluster_diversity'] >= 0).all() if 'cluster_diversity' in df_summary.columns else True, '')
chk('error_count >=1',       (df_summary['error_count'] >= 1).all() if 'error_count' in df_summary.columns else True, '')

n_pass  = sum(c['passed'] for c in checks)
n_total = len(checks)
print(f'\nValidation: {n_pass}/{n_total} checks passed')

failed = [c['name'] for c in checks if not c['passed']]
if failed:
    raise RuntimeError(f'NB11 validation failed: {failed}')

print('NB11 COMPLETE — artifacts ready for NB12 (solution embeddings).')